# 📚 Data Cleaning
This notebook executes the data cleaning

Import Library

In [ ]:
# 01_data_cleaning.ipynb
import pandas as pd
import numpy as np
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

1️⃣ Load ERP data

In [ ]:
# Path to your ERP Excel file
erp_path = "data/inputs/erp_data_combined.xlsx"
erp = pd.read_excel(erp_path)

# Inspect first rows
erp.head()

,Order_Year,Order_Number,Customer_Order_Number,Customer_Number,Customer_Name,Product_Item_Number,Product_Item_Group,Product_Item_Name,Order_Date,Planned_Delivery_Date,Confirmed_Delivery_Date,Status,Original_Delivery_Date,Last_Delivery_Date,Ordered_Quantity,Delivery_Quantity,Quantity_Left,Unit,Order_Amount,Group_Name,Unit_Price
0,2015,CO500007,NaN,9999,MUUT ASIAKKAAT,5002,5002,Tappikiinnitteinen Sylinteri,2015-04-07,2015-04-08,2015-04-08,laskutettu,2015-04-08,2015-04-07,1,1.0,0,kpl,282.26,Tappikiinn.Sylin.,282.26
1,2015,CO500008,NaN,9999,MUUT ASIAKKAAT,N02460585,5002,Sylinteri 60/40-870 6040870,2015-04-07,2015-04-07,2015-04-07,laskutettu,2015-04-07,2015-04-07,1,1.0,0,kpl,240.00,Tappikiinn.Sylin.,240.00
2,2015,GR500003,NaN,21822,OUTOKUMMUN METALLI,V100001,5002,"0hh10509005502470479 90/55-247,5 a479",2015-02-11,2015-03-12,2015-03-12,toimitettu,2015-03-12,2015-03-25,2,2.0,0,kpl,370.46,Tappikiinn.Sylin.,185.23
3,2015,GR500003,NaN,21822,OUTOKUMMUN METALLI,F673812,5002,0pp08007004001720386 70/40-172 a386,2015-02-11,2015-03-12,2015-03-12,toimitettu,2015-03-12,2015-04-07,1,1.0,0,kpl,140.86,Tappikiinn.Sylin.,140.86
4,2015,GR500015,NaN,9999,MUUT ASIAKKAAT,08005006500005,5005,Sylinteri 80/50-650 A824,2015-05-22,2015-05-22,2015-05-22,toimitettu,2015-05-22,2015-05-22,1,1.0,0,kpl,75.00,Erikoissylin.,75.00


2️⃣ Select relevant columns and rename

In [ ]:
# keep the essential columns
erp = erp[[
    'Order_Year','Order_Number','Customer_Number','Customer_Name',
    'Product_Item_Number','Product_Item_Group','Product_Item_Name',
    'Order_Date','Planned_Delivery_Date','Confirmed_Delivery_Date',
    'Ordered_Quantity','Order_Amount','Unit_Price'
]].copy()

# rename for consistency
erp.rename(columns={
    'Order_Year': 'year',
    'Order_Number': 'order_no',
    'Customer_Number': 'customer_id',
    'Customer_Name': 'customer_name',
    'Product_Item_Number': 'product_id',
    'Product_Item_Group': 'product_group',
    'Product_Item_Name': 'product_name',
    'Order_Date': 'order_date',
    'Planned_Delivery_Date': 'planned_delivery_date',
    'Confirmed_Delivery_Date': 'confirmed_delivery_date',
    'Ordered_Quantity': 'ordered_qty',
    'Order_Amount': 'order_amount',
    'Unit_Price': 'unit_price'
}, inplace=True)

3️⃣ Convert dates and clean values

In [ ]:
# Convert to datetime
for c in ['order_date','planned_delivery_date','confirmed_delivery_date']:
    erp[c] = pd.to_datetime(erp[c], errors='coerce')

# Filter out negative or zero quantities
erp = erp.loc[erp['ordered_qty'] > 0].copy()

# Drop missing key identifiers
erp.dropna(subset=['order_date','customer_id','product_id'], inplace=True)

4️⃣ Create monthly index

In [ ]:
erp['YearMonth'] = erp['order_date'].dt.to_period('M').astype(str)
erp['Date'] = pd.to_datetime(erp['YearMonth'] + '-01')

erp[['Date','customer_id','product_id','ordered_qty','order_amount']].head()

,Date,customer_id,product_id,ordered_qty,order_amount
0,2015-04-01,9999,5002,1,282.26
1,2015-04-01,9999,N02460585,1,240.00
2,2015-02-01,21822,V100001,2,370.46
3,2015-02-01,21822,F673812,1,140.86
4,2015-05-01,9999,08005006500005,1,75.00


5️⃣ Save cleaned ERP

In [ ]:
os.makedirs('data/cleaned', exist_ok=True)
erp.to_csv('data/cleaned/erp_cleaned.csv', index=False)
print("✅ Saved: data/cleaned/erp_cleaned.csv")

✅ Saved: data/cleaned/erp_cleaned.csv


6️⃣ Load & clean external macro data

In [ ]:
ext_path = "data/inputs/external_data_combined.xlsx"   # adjust path if needed
external = pd.read_excel(ext_path)
print("Columns:", external.columns.tolist())
print(external.head())

external.rename(columns={'Order_Month': 'Date'}, inplace=True)

# --- Convert to datetime and ensure monthly frequency ---
external['Date'] = pd.to_datetime(external['Date'])
external = (
    external.set_index('Date')
             .resample('MS')
             .ffill()   # forward-fill for quarterly data
             .reset_index()
)

# --- Add missingness masks for each variable (useful for deep models) ---
for col in external.columns:
    if col != 'Date':
        mask_col = col + "_is_real"
        external[mask_col] = external[col].notna().astype(int)

# --- Save cleaned file ---
os.makedirs('data/cleaned', exist_ok=True)
external.to_csv('data/cleaned/external_cleaned.csv', index=False)

print("✅ Saved data/cleaned/external_cleaned.csv")
external.head()

Columns: ['Order_Month', 'Food_Price_Index', 'Electricity_Price', 'ECB_Inflation', 'ECB_Interest_Rate', 'FEDFUNDS_Value', 'Purchase_Index', 'Total_New_Orders_Value', 'Steel_Price']
  Order_Month  Food_Price_Index  Electricity_Price  ECB_Inflation  ECB_Interest_Rate  FEDFUNDS_Value  Purchase_Index  \
0  2021-01-01            102.86           0.083143            0.9             -0.378            0.09             NaN   
1  2021-02-01            103.18           0.086634            0.9             -0.218            0.08             NaN   
2  2021-03-01            103.34           0.090124            1.3             -0.126            0.07             NaN   
3  2021-04-01            103.48           0.093615            1.6             -0.079            0.07             NaN   
4  2021-05-01            103.17           0.097106            2.0              0.047            0.06             NaN   

   Total_New_Orders_Value  Steel_Price  
0            10002.500000        213.1  
1            102

,Date,Food_Price_Index,Electricity_Price,ECB_Inflation,ECB_Interest_Rate,FEDFUNDS_Value,Purchase_Index,Total_New_Orders_Value,Steel_Price,Food_Price_Index_is_real,Electricity_Price_is_real,ECB_Inflation_is_real,ECB_Interest_Rate_is_real,FEDFUNDS_Value_is_real,Purchase_Index_is_real,Total_New_Orders_Value_is_real,Steel_Price_is_real
0,2021-01-01,102.86,0.083143,0.9,-0.378,0.09,NaN,10002.500000,213.1,1,1,1,1,1,0,1,1
1,2021-02-01,103.18,0.086634,0.9,-0.218,0.08,NaN,10203.100000,193.9,1,1,1,1,1,0,1,1
2,2021-03-01,103.34,0.090124,1.3,-0.126,0.07,NaN,10403.700000,210.5,1,1,1,1,1,0,1,1
3,2021-04-01,103.48,0.093615,1.6,-0.079,0.07,NaN,10604.300000,208.7,1,1,1,1,1,0,1,1
4,2021-05-01,103.17,0.097106,2.0,0.047,0.06,NaN,10544.733333,236.2,1,1,1,1,1,0,1,1


### 🎉 Pipeline completed successfully!
You can now proceed to notebook 02 for feature Engineering